# 2. Evaluate a population at a given age

When we consider a star cluster as a whole, we are often interested in evaluating its overall properties. For example, the total power it emits as radiation at a given age.

In this tutorial we will evaluate two population-level properties at a specified age: its **total bolometric luminosity** and its **luminosity-weighted mean effective temperature**.

In [5]:
import astropy.units as u
import arsenal_gear as ag

We start by creating a simple stellar population (SSP), retaining the settings from the first tutorial.

In [6]:
ssp_parameters = {
    "type": "single",
    "Mtot": 1e5 * u.Msun,
    "tform": 0 * u.Myr,
    "metallicity": 0.0,
    "rotation": 0.0,
    "imf": "kroupa2001",
    "mmin": 0.1 * u.Msun,
    "mmax": 100 * u.Msun,
    "discrete": True,
    "seed": 42,
}

sp = ag.SynthPop(pop1=ssp_parameters, interp_op="iso", vvcrit="0.0")
pop=sp.form.subpops[0]

The population has two complementary parts: `sp.form.subpops[0]` stores the stellar content (formation parameters and sampled initial masses from the IMF), while `sp.evol.ses[0]` provides its stellar-evolution interface. We will call it `se`.

In [7]:
se = sp.evol.ses[0]

print("Evolution interface:", type(se).__name__)
print("Model reader:", type(se.reader).__name__)

Evolution interface: IsochroneInterpolator
Model reader: MISTReader


Here, `se` is an `IsochroneInterpolator` using a `MISTReader` to access the precomputed stellar-evolution models by **MIST** (MESA Isochrones and Stellar Tracks). **Isochrones** describe stars with different initial masses at a common age and initial chemical composition (CHECK DEDICATED TUTORIAL). By selecting `interp_op='iso', ARSENAL interpolates the precomputed isochrones to the requested age and then obtains stellar properties at the sampled initial masses. 

However, these models are not without limitations. MIST’s nominal mass coverage begins at approximately $0.1\,M_\odot$. Stars with masses outside of this coverage cannot be evolved by these models, so ARSENAL masks and excludes their contributions.

To evaluate the SSP at an age of $10\,\mathrm{Myr}$, we should bare in mind that population methods take, as the argument, a time $t$ measured relative to the same origin as the formation time. The age of this component is therefore:

$$
\tau = t - t_{\mathrm{form}}.
$$

Since we formed the population at $t_{\mathrm{form}}=0$, time and age have the same numerical value here. Still, this will become relevant as we start working with composite systems where populations might have formed at different times.

In [8]:
age = 10 * u.Myr
t = pop.tform + age

We can now query the population directly to obtain its properties.

## Bolometric luminosity

The bolometric luminosity is the energy emitted as radiation per unit time, integrated over all wavelengths. 

For a discrete population, ARSENAL evaluates the luminosity at each sampled initial mass and sums them:

$$
L_{\mathrm{bol}}(\tau) = \sum_i L_i(\tau).
$$

The result is the total bolometric luminosity expressed in units of solar luminosity, $L_\odot$.

In [9]:
lbol=sp.lbol(t)
print(f"Bolometric luminosity: {lbol:.3e}")

Bolometric luminosity: 1.859e+07 solLum


## Luminosity-weighted effective temperature

For an individual star, the effective temperature is the temperature of a black body with the same bolometric luminosity as the star. It's related to its bolometric luminosity and radius through $L_i=4\pi R_i^2\sigma_{\mathrm{SB}}T_{\mathrm{eff},i}^4$, where $\sigma_{\mathrm{SB}}$ is the Stefan–Boltzmann constant.

The population contains stars with different temperatures. ARSENAL reports a luminosity-weighted mean:

$$
\left\langle T_{\mathrm{eff}}\right\rangle_L(\tau)
= \frac{\sum_i L_i(\tau)\,T_{\mathrm{eff},i}(\tau)}{\sum_i L_i(\tau)}.
$$

Thus, more luminous stars contribute more strongly to this mean, even if they are few in number.

In [11]:
teff=sp.teff(t)
print(f"Effective temperature: {teff:.3e}")

Effective temperature: 1.984e+04 K


In the next tutorial we will evaluate this same population at a sequence of ages and interpret its luminosity and temperature histories.